# AE architecture, depth, and latent-dimension study

This notebook is a validation-only dashboard for Conv1D and Conv2D autoencoders with latent dimensions 2, 4, 6, 8, and 10. The direct models and commands live in `Code/iaflow/autoencoder`, while family-independent data, evaluation, and runtime services live in `Code/iaflow/core`. Every model compresses each `log10(A_theta)` surface from `(31, 101)` without skip connections. The test split is not read anywhere in this notebook.

In [ ]:
import json
import sys
from pathlib import Path

import h5py
import numpy as np
import torch
from matplotlib import pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Run this notebook from within the IAFlowCloud repository.')
    PROJECT_ROOT = PROJECT_ROOT.parent
code_path = PROJECT_ROOT / 'Code'
if str(code_path) not in sys.path:
    sys.path.insert(0, str(code_path))

from iaflow.autoencoder.artifacts import load_compatible_autoencoder_checkpoint
from iaflow.autoencoder.config import config_to_dict, load_experiment_template
from iaflow.comparison import load_complete_validation_record
from iaflow.core.data import CACHE_FORMAT_VERSION, CachedSurfaceDataset, check_surface_cache
from iaflow.autoencoder.model import build_autoencoder
from iaflow.core.runs import completed_run, discover_run_directories

LATENT_DIMENSIONS = (2, 4, 6, 8, 10)
SELECTED_LATENT_DIMENSION = 6
ARCHITECTURE = 'Conv1D'
DEPTH = 'Depth04'
DENSE_HIDDEN_BY_DEPTH = {
    'Depth03': (256, 64, 16),
    'Depth04': (512, 256, 64, 16),
    'Depth05': (768, 512, 256, 64, 16),
}
EXPECTED_DENSE_HIDDEN = DENSE_HIDDEN_BY_DEPTH[DEPTH]
CONFIG_PATH = PROJECT_ROOT / 'Config' / 'NLA' / 'AE' / ARCHITECTURE / f'{DEPTH}.yaml'
template = load_experiment_template(CONFIG_PATH, project_root=PROJECT_ROOT)
EXPERIMENT_CONFIGS = {
    latent_dim: template.resolve(
        latent_dim,
        Path(template.output.root_directory) / f'Latent{latent_dim:02d}' / 'NotebookPreview',
    )
    for latent_dim in LATENT_DIMENSIONS
}

config = EXPERIMENT_CONFIGS[SELECTED_LATENT_DIMENSION]
print('PyTorch:', torch.__version__)
print('Project:', PROJECT_ROOT)
print('Configured latent dimensions:', LATENT_DIMENSIONS)
print('Selected latent dimension:', config.model.latent_dim)

## 1. Validate the source-ordered cache

Preparation writes one memory-mappable source-ordered array, not one copy per split. The HDF5 split indices remain authoritative and normalization is estimated only from its training rows.

In [ ]:
cache_directory = config.resolve_path(config.data.cache_directory)
metadata_path = cache_directory / 'Metadata.json'
if metadata_path.exists():
    metadata = check_surface_cache(config)
    print(json.dumps(metadata, indent=2))
else:
    metadata = None
    print('Cache not prepared. Run:')
    print(f'iaflow-prepare-data --config {CONFIG_PATH}')

## 2. Inspect the configured bottleneck

The 31 redshift values are channels and convolution runs along the 101-point wavenumber axis. Every reconstruction passes through the selected latent bottleneck, while all other architecture settings remain fixed across the study.

In [ ]:
model = build_autoencoder(config.model, config.data.input_shape)
print(json.dumps(model.architecture_summary(), indent=2))
try:
    from torchinfo import summary
    summary(model, input_size=(2, *config.data.input_shape), device='cpu')
except ImportError:
    print(model)

## 3. Train from the command line

Long jobs run through the script so they produce complete artifacts and can resume independently of the notebook UI.

In [ ]:
for latent_dim in LATENT_DIMENSIONS:
    print(f'latent {latent_dim:02d}: iaflow-train-autoencoder --config {CONFIG_PATH} --latent-dim {latent_dim}')
print('Validation-only smoke option: add --epochs 2 --maximum-train-samples 1024 --maximum-validation-samples 256')

## 4. Review validation history and runtime

In [ ]:
def run_matches_configuration(run_directory, latent_dim):
    resolved_config_path = run_directory / 'ResolvedConfig.json'
    if not completed_run(run_directory) or not (run_directory / 'History.json').is_file():
        return False
    
    resolved_config = json.loads(resolved_config_path.read_text())
    model_config = resolved_config.get('model', {})
    data_config = resolved_config.get('data', {})
    output_config = resolved_config.get('output', {})
    return (
        model_config.get('name') == ARCHITECTURE
        and Path(output_config.get('root_directory', '')).name == DEPTH
        and model_config.get('latent_dim') == latent_dim
        and tuple(model_config.get('dense_hidden', ())) == EXPECTED_DENSE_HIDDEN
        and data_config.get('batch_size') == 512
        and data_config.get('evaluation_batch_size') == 512
    )


def selected_run_directory(latent_dim):
    latent_roots = [
        template.resolve_path(template.output.root_directory) / f'Latent{latent_dim:02d}'
    ]
    if ARCHITECTURE == 'Conv1D' and DEPTH == 'Depth03':
        latent_roots.append(
            PROJECT_ROOT / 'Runs' / 'NLA' / 'AE' / 'Conv1D' / f'Latent{latent_dim:02d}'
        )
    
    for latent_root in latent_roots:
        for candidate in reversed(discover_run_directories(latent_root)):
            if run_matches_configuration(candidate, latent_dim):
                return candidate
    return None

RUN_DIRECTORY = selected_run_directory(SELECTED_LATENT_DIMENSION)

if RUN_DIRECTORY is None or not (RUN_DIRECTORY / 'Best.pt').is_file():
    RUN_DIRECTORY = None
    print('No completed compatible run is registered yet.')
else:
    history = json.loads((RUN_DIRECTORY / 'History.json').read_text())
    epochs = [row['epoch'] for row in history]
    train_loss = [row['train_loss'] for row in history]
    validation_loss = [row['validation']['normalized_mse'] for row in history]
    validation_variance = [row['validation']['variance_recovered'] for row in history]
    training_seconds = [row.get('training_seconds', np.nan) for row in history]
    validation_seconds = [row.get('validation_seconds', np.nan) for row in history]
    checkpoint_seconds = [row.get('checkpoint_seconds', np.nan) for row in history]
    figure, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].semilogy(epochs, train_loss, label='training objective')
    axes[0].semilogy(epochs, validation_loss, label='validation MSE')
    axes[0].set(xlabel='epoch', ylabel='normalized loss')
    axes[0].legend()
    axes[1].plot(epochs, validation_variance)
    axes[1].axhline(0.999, color='black', linestyle='--', label='99.9% target')
    axes[1].set(xlabel='epoch', ylabel='validation variance recovered')
    axes[1].legend()
    axes[2].plot(epochs, training_seconds, label='training')
    axes[2].plot(epochs, validation_seconds, label='validation')
    axes[2].plot(epochs, checkpoint_seconds, label='checkpoint')
    axes[2].set(xlabel='epoch', ylabel='seconds')
    axes[2].legend()
    figure.tight_layout()
    plt.show()

## 5. Representative and worst-case validation reconstructions

The following diagnostics inspect only a deterministic prefix of the validation split and display its best, median, and worst reconstruction by normalized MSE.

In [ ]:
diagnostic_count = 512
validation_data = None
validation_latent = None
if RUN_DIRECTORY is not None:
    trained_model, normalization, checkpoint, _ = load_compatible_autoencoder_checkpoint(
        RUN_DIRECTORY / 'Best.pt', config
    )
    validation_data = CachedSurfaceDataset(config, 'validation')
    diagnostic_count = min(diagnostic_count, len(validation_data))
    targets = torch.stack([validation_data[index] for index in range(diagnostic_count)])
    with torch.inference_mode():
        validation_latent = trained_model.encode(targets).cpu().numpy()
        reconstructions = trained_model(targets).cpu().numpy()
    target_values = targets.numpy()
    surface_mse = np.mean((reconstructions - target_values) ** 2, axis=(1, 2))
    order = np.argsort(surface_mse)
    selected = [order[0], order[len(order) // 2], order[-1]]
    labels = ['best', 'median', 'worst']
    figure, axes = plt.subplots(3, 3, figsize=(13, 11), constrained_layout=True)
    for row, (index, label) in enumerate(zip(selected, labels)):
        truth = normalization.denormalize(target_values[index])
        prediction = normalization.denormalize(reconstructions[index])
        residual = prediction - truth
        limit = max(float(np.max(np.abs(residual))), np.finfo(np.float32).eps)
        images = [
            axes[row, 0].imshow(truth, aspect='auto', origin='lower'),
            axes[row, 1].imshow(prediction, aspect='auto', origin='lower'),
            axes[row, 2].imshow(residual, aspect='auto', origin='lower', cmap='coolwarm', vmin=-limit, vmax=limit),
        ]
        axes[row, 0].set_title(f'{label}: truth log10(A_theta)')
        axes[row, 1].set_title(f'reconstruction; MSE={surface_mse[index]:.3e}')
        axes[row, 2].set_title('residual')
        for axis, image in zip(axes[row], images):
            axis.set(xlabel='wavenumber index', ylabel='redshift channel')
            figure.colorbar(image, ax=axis, shrink=0.75)
    plt.show()

## 6. Latent geometry and nuisance correlations

The first plot projects the selected latent representation onto its first two coordinates for visualization. The correlation matrix then shows how every learned coordinate relates to the 13 sampled NLA shape parameters.

In [ ]:
if validation_latent is not None:
    figure, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].scatter(validation_latent[:, 0], validation_latent[:, 1], s=8, alpha=0.45)
    axes[0].set(xlabel='latent 1', ylabel='latent 2', title='first two validation coordinates')
    axes[1].hist(validation_latent[:, 0], bins=35, density=True)
    axes[1].set(xlabel='latent 1', ylabel='density')
    axes[2].hist(validation_latent[:, 1], bins=35, density=True)
    axes[2].set(xlabel='latent 2', ylabel='density')
    figure.tight_layout()
    plt.show()
    
    source_path = config.resolve_path(config.data.source_path)
    source_indices = validation_data.source_indices[:diagnostic_count]
    with h5py.File(source_path, 'r') as source:
        nuisance_values = source['parameters/values'][source_indices]
        nuisance_names = [
            value.decode() if isinstance(value, bytes) else str(value)
            for value in source['parameters/names'][:]
        ]
    latent_dim = validation_latent.shape[1]
    joined = np.column_stack([validation_latent, nuisance_values])
    correlations = np.nan_to_num(
        np.corrcoef(joined, rowvar=False)[:latent_dim, latent_dim:]
    )
    figure, axis = plt.subplots(figsize=(12, max(3.2, 0.45 * latent_dim)))
    image = axis.imshow(correlations, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
    axis.set(
        yticks=np.arange(latent_dim),
        yticklabels=[f'latent {index + 1}' for index in range(latent_dim)],
    )
    axis.set(xticks=np.arange(len(nuisance_names)), xticklabels=nuisance_names)
    axis.tick_params(axis='x', rotation=55)
    figure.colorbar(image, ax=axis, label='Pearson correlation')
    figure.tight_layout()
    plt.show()

## 7. Identical-split PCA comparison

`PCA.ipynb` exports complete-validation metrics for ranks 2, 4, 6, 8, and 10 using the same `log10` target, training mean, global RMS, and stored validation indices. This dashboard accepts only current-schema `ValidationMetrics.json` artifacts and reports the variance-recovery percentage-point gain plus fractional error reductions relative to matched-rank PCA. Positive reductions mean lower autoencoder error.

In [ ]:
pca_metrics_path = PROJECT_ROOT / 'Data' / 'NLA' / 'PCA' / 'PCAValidationMetrics.json'
run_summaries = {}
validation_records = {}
for latent_dim in LATENT_DIMENSIONS:
    run_directory = selected_run_directory(latent_dim)
    if run_directory is None:
        continue
    summary_path = run_directory / 'Summary.json'
    if summary_path.exists():
        run_summaries[latent_dim] = json.loads(summary_path.read_text())
    validation_record = load_complete_validation_record(PROJECT_ROOT, run_directory)
    if validation_record is not None:
        validation_records[latent_dim] = validation_record

if pca_metrics_path.exists():
    pca_metrics = json.loads(pca_metrics_path.read_text())
    print(
        f"{'Dimension':>9s} {'PCA variance':>14s} {'AE variance':>14s} "
        f"{'VR gain (pp)':>12s} {'MSE reduction':>14s} "
        f"{'Mean rel. reduction':>20s} {'p99 max reduction':>19s}"
    )
    print('-' * 111)
    for latent_dim in LATENT_DIMENSIONS:
        pca_rank = pca_metrics['ranks'][str(latent_dim)]
        if latent_dim not in validation_records:
            print(
                f'{latent_dim:9d} {pca_rank["variance_recovered"]:14.8%} '
                f'{"pending":>14s} {"pending":>12s} '
                f'{"pending":>14s} {"pending":>20s} {"pending":>19s}'
            )
            continue
        
        ae_metrics, comparison = validation_records[latent_dim]
        reductions = comparison['autoencoder_fractional_error_reduction']
        print(
            f'{latent_dim:9d} {pca_rank["variance_recovered"]:14.8%} '
            f'{ae_metrics["variance_recovered"]:14.8%} '
            f'{comparison["variance_recovered_percentage_point_gain"]:12.5f} '
            f'{reductions["log10_mse"]:14.4%} '
            f'{reductions["mean_relative_error"]:20.4%} '
            f'{reductions["surface_relative_maximum_p99"]:19.4%}'
        )
    
    completed_dimensions = sorted(validation_records)
    if completed_dimensions:
        pca_variance = [
            pca_metrics['ranks'][str(latent_dim)]['variance_recovered']
            for latent_dim in completed_dimensions
        ]
        ae_variance = [
            validation_records[latent_dim][0]['variance_recovered']
            for latent_dim in completed_dimensions
        ]
        pca_log10_mse = [
            pca_metrics['ranks'][str(latent_dim)]['log10_mse']
            for latent_dim in completed_dimensions
        ]
        ae_log10_mse = [
            validation_records[latent_dim][0]['log10_mse']
            for latent_dim in completed_dimensions
        ]
        figure, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(completed_dimensions, pca_variance, marker='o', label='PCA')
        axes[0].plot(completed_dimensions, ae_variance, marker='o', label='autoencoder')
        axes[0].axhline(0.999, color='black', linestyle='--', label='99.9% target')
        axes[0].set(xlabel='latent dimension', ylabel='validation variance recovered')
        axes[0].legend()
        axes[1].semilogy(completed_dimensions, pca_log10_mse, marker='o', label='PCA')
        axes[1].semilogy(completed_dimensions, ae_log10_mse, marker='o', label='autoencoder')
        axes[1].set(xlabel='latent dimension', ylabel='validation log10 MSE')
        axes[1].legend()
        figure.tight_layout()
        plt.show()
else:
    print('Run PCA.ipynb to regenerate the matched-rank comparison file.')

## 8. Freeze, test once, and export latents

After all choices are frozen, run the final test command once with `--confirm-final-test`. A smoke run must use `--split validation` instead.

```bash
iaflow-evaluate-autoencoder --run-directory Runs/NLA/AE/<architecture>/<depth>/LatentXX/<final-run> --split test --confirm-final-test
iaflow-export-latents --run-directory Runs/NLA/AE/<architecture>/<depth>/LatentXX/<final-run> --include-test
```

## Final consistency checks

In [ ]:
reference_values = config_to_dict(EXPERIMENT_CONFIGS[LATENT_DIMENSIONS[0]])
reference_values['model'].pop('latent_dim')
reference_values['output'].pop('run_directory')

for latent_dim, experiment_config in EXPERIMENT_CONFIGS.items():
    assert experiment_config.model.name == ARCHITECTURE
    assert experiment_config.model.latent_dim == latent_dim
    assert tuple(experiment_config.model.dense_hidden) == EXPECTED_DENSE_HIDDEN
    assert experiment_config.data.input_shape == (31, 101)
    assert experiment_config.data.batch_size == 512
    assert experiment_config.data.evaluation_batch_size == 512
    assert CONFIG_PATH.is_file()
    assert experiment_config.output.root_directory.endswith(DEPTH)
    assert f'Latent{latent_dim:02d}' in experiment_config.output.run_directory
    
    comparison_values = config_to_dict(experiment_config)
    comparison_values['model'].pop('latent_dim')
    comparison_values['output'].pop('run_directory')
    assert comparison_values == reference_values

assert config.data.input_shape == (31, 101)
if metadata is not None:
    assert metadata['cache_format_version'] == CACHE_FORMAT_VERSION
    assert tuple(metadata['input_shape']) == config.data.input_shape
    assert 'split_indices' not in metadata

for summary in run_summaries.values():
    assert not summary['test_split_used_during_training']
print('Autoencoder notebook consistency checks passed.')